In [2]:
import datetime
from datetime import timedelta

import sqlite3
import numpy as np
import matplotlib.pyplot as plt

In [23]:
weekStarts   = []
weekEnds     = []
sbWeekStarts = []
sbWeekEnds   = []

days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

seasonStart = datetime.datetime(2025, 3, 18)
seasonEnd   = datetime.datetime(2025, 9, 21)

seasonStartDayNumber = seasonStart.weekday()

print(seasonStartDayNumber)
print(days[seasonStartDayNumber])

daysToAdd = 0

daysToAdd = timedelta(days = 6 - seasonStartDayNumber)

endFirstWeek = seasonStart + daysToAdd

endFirstWeekNumber = endFirstWeek.weekday()

formattedCurWeekStart = seasonStart   .strftime("%Y-%m-%d")
formattedCurWeekEnd   = endFirstWeek  .strftime("%Y-%m-%d")

formattedSbCurWeekStart = seasonStart   .strftime("%Y%m%d")
formattedSbCurWeekEnd   = endFirstWeek  .strftime("%Y%m%d")

weekStarts.append(formattedCurWeekStart )
weekEnds  .append(formattedCurWeekEnd   )

sbWeekStarts.append(formattedSbCurWeekStart)
sbWeekEnds  .append(formattedSbCurWeekEnd  )

print(f"week 1: {formattedCurWeekStart} -> {formattedCurWeekEnd}")

curWeekStart = endFirstWeek + timedelta(days=1)
curWeekEnd   = curWeekStart + timedelta(days=6)

i = 2

while curWeekStart <= seasonEnd:
    formattedCurWeekStart = curWeekStart.strftime("%Y-%m-%d")
    formattedCurWeekEnd   = curWeekEnd  .strftime("%Y-%m-%d")

    formattedSbCurWeekStart = curWeekStart.strftime("%Y%m%d")
    formattedSbCurWeekEnd   = curWeekEnd  .strftime("%Y%m%d")
    
    print(f"week {i}: {formattedCurWeekStart} -> {formattedCurWeekEnd}")

    weekStarts.append(formattedCurWeekStart)
    weekEnds  .append(formattedCurWeekEnd  )

    sbWeekStarts.append(formattedSbCurWeekStart)
    sbWeekEnds  .append(formattedSbCurWeekEnd  )
    
    curWeekStart = curWeekEnd   + timedelta(days=1)
    curWeekEnd   = curWeekStart + timedelta(days=6)

    if curWeekEnd > seasonEnd:
        curWeekEnd = seasonEnd

    i = i + 1

1
Tuesday
week 1: 2025-03-18 -> 2025-03-23
week 2: 2025-03-24 -> 2025-03-30
week 3: 2025-03-31 -> 2025-04-06
week 4: 2025-04-07 -> 2025-04-13
week 5: 2025-04-14 -> 2025-04-20
week 6: 2025-04-21 -> 2025-04-27
week 7: 2025-04-28 -> 2025-05-04
week 8: 2025-05-05 -> 2025-05-11
week 9: 2025-05-12 -> 2025-05-18
week 10: 2025-05-19 -> 2025-05-25
week 11: 2025-05-26 -> 2025-06-01
week 12: 2025-06-02 -> 2025-06-08
week 13: 2025-06-09 -> 2025-06-15
week 14: 2025-06-16 -> 2025-06-22
week 15: 2025-06-23 -> 2025-06-29
week 16: 2025-06-30 -> 2025-07-06
week 17: 2025-07-07 -> 2025-07-13
week 18: 2025-07-14 -> 2025-07-20
week 19: 2025-07-21 -> 2025-07-27
week 20: 2025-07-28 -> 2025-08-03
week 21: 2025-08-04 -> 2025-08-10
week 22: 2025-08-11 -> 2025-08-17
week 23: 2025-08-18 -> 2025-08-24
week 24: 2025-08-25 -> 2025-08-31
week 25: 2025-09-01 -> 2025-09-07
week 26: 2025-09-08 -> 2025-09-14
week 27: 2025-09-15 -> 2025-09-21


In [4]:
def get_percentile(sortedList, value):
    arr = np.array(sortedList)
    count_less_than = np.sum(arr < value)
    percentile = (count_less_than / len(arr)) * 100
    return percentile

In [5]:
def normalize(num, maxValue, minValue):
    if maxValue == 0 and minValue == 0:
        return 0
    if maxValue == minValue:
        return 0
    
    normalizedValue = (num - minValue) / (maxValue - minValue)

    return normalizedValue

In [37]:
teamToGamesPlayed = {}

homeGameQueryPartOne = "SELECT hometeam as team,COUNT(gid) as homeGames from games where date between \"2025-03-18\" AND \""
homeGameQueryPartTwo = "\" group by games.hometeam"

awayGameQueryPartOne = "SELECT visteam as team,COUNT(gid) as awayGames from games where date between \"2025-03-18\" AND \""
awayGameQueryPartTwo = "\" group by games.visteam"

connection = sqlite3.connect('player_batting.db')
cursor     = connection.cursor()

for i in range(len(weekEnds)):
    teamToGamesPlayed[i] = {}
    
    weekEnd = weekEnds[i]

    homeGameQuery = homeGameQueryPartOne + weekEnd + homeGameQueryPartTwo

    cursor.execute(homeGameQuery)

    rows = cursor.fetchall()
    
    for row in rows:
        team = row[0]
        games = int(row[1])
    
        teamToGamesPlayed[i][team] = games

    awayGameQuery = awayGameQueryPartOne + weekEnd + awayGameQueryPartTwo
    
    cursor.execute(awayGameQuery)
    
    rows = cursor.fetchall()
    
    for row in rows:
        team = row[0]
        games = int(row[1])
    
        if team in teamToGamesPlayed[i]:
            teamToGamesPlayed[i][team] = teamToGamesPlayed[i][team] + games
        else:
            teamToGamesPlayed[i][team] = games

cursor    .close()
connection.close()

In [1]:
# for i in range(len(weekStarts)):
#     print(f"Week Number: {i + 1}")
#     for key in teamToGamesPlayed[i]:
#         print(f"{key}: {teamToGamesPlayed[i][key]}")

In [10]:
hrQueryPartOne = "SELECT id,b_hr FROM batters WHERE b_pa != \"0\" AND gametype = \"regular\" AND date BETWEEN \""
hrQueryPartTwo = "\" AND \""

addGradeStatement = "INSERT INTO weekly_grades (week_number,week_start,week_end,player_id,stat,num,percentile,season) VALUES("

#for i in range(len(weekStarts)):
for i in range(25,27):
    connection = sqlite3.connect('player_batting.db')
    cursor     = connection.cursor()

    weekStart = weekStarts[i]
    weekEnd   = weekEnds  [i]

    hrQuery = hrQueryPartOne + weekStart + hrQueryPartTwo + weekEnd + "\""

    cursor.execute(hrQuery)

    rows = cursor.fetchall()

    weeklyPlayerToHomeRuns = {}
    
    for row in rows:
        playerId = row[0]
        hrs      = int(row[1])

        if playerId in weeklyPlayerToHomeRuns:
            curHrs = weeklyPlayerToHomeRuns[playerId]
            weeklyPlayerToHomeRuns[playerId] = curHrs + hrs
        else:
            weeklyPlayerToHomeRuns[playerId] = hrs

    cursor    .close()
    connection.close()

    weeklyHomeRunList = []

    for key in weeklyPlayerToHomeRuns:
        hrs = weeklyPlayerToHomeRuns[key]
        weeklyHomeRunList.append(hrs)

    weeklyHomeRunList.sort()

    for key in weeklyPlayerToHomeRuns:
        hrs = weeklyPlayerToHomeRuns[key]

        percentile = get_percentile(weeklyHomeRunList, hrs)

        formattedAddGradeStatement = addGradeStatement + str(i+1) + "," + "\"" + weekStart + "\",\"" + weekEnd + "\",\"" + key + "\",\"HOME_RUN\"," + str(hrs) + "," + str(percentile) + ", " + "2025" + ")"

        connection = sqlite3.connect('player_batting.db')
        cursor     = connection.cursor()

        cursor.execute(formattedAddGradeStatement)
        connection.commit()
        
        cursor    .close()
        connection.close()        

In [11]:
hrQuery = "SELECT id,sum(b_hr) FROM batters WHERE b_pa != \"0\" AND gametype = \"regular\" AND date BETWEEN \"2025-03-18\" AND \""

addGradeStatement = "INSERT INTO aggregate_weekly_grades (week_number,week_start,week_end,player_id,stat,aggregate_num,percentile,season) VALUES("

for i in range(25,27):
    aggregatePlayerToHomeRuns = {}
    
    connection = sqlite3.connect('player_batting.db')
    cursor     = connection.cursor()

    weekStart = weekStarts[i]
    weekEnd   = weekEnds  [i]

    playerHrQuery = hrQuery + weekEnd + "\" GROUP BY id"

    cursor.execute(playerHrQuery)

    rows = cursor.fetchall()
    
    for row in rows:
        playerId = row[0]
        hrs      = int(row[1])
        
        aggregatePlayerToHomeRuns[playerId] = hrs

    cursor    .close()
    connection.close()

    aggregateHomeRunList = []

    for key in aggregatePlayerToHomeRuns:
        hrs = aggregatePlayerToHomeRuns[key]
        aggregateHomeRunList.append(hrs)

    aggregateHomeRunList.sort()

    for key in aggregatePlayerToHomeRuns:
        hrs = aggregatePlayerToHomeRuns[key]

        percentile = get_percentile(aggregateHomeRunList, hrs)
        
        formattedAddGradeStatement = addGradeStatement + str(i+1) + "," + "\"" + weekStart + "\",\"" + weekEnd + "\",\"" + key + "\",\"HOME_RUN\"," + str(hrs) + "," + str(percentile) + ", " + "2025" + ")"

        connection = sqlite3.connect('player_batting.db')
        cursor     = connection.cursor()

        cursor.execute(formattedAddGradeStatement)
        connection.commit()
        
        cursor    .close()
        connection.close()        

In [12]:
sbQueryPartOne = "SELECT id,b_sb FROM batters WHERE gametype = \"regular\" AND date BETWEEN \""
sbQueryPartTwo = "\" AND \""

addGradeStatement = "INSERT INTO weekly_grades (week_number,week_start,week_end,player_id,stat,num,percentile,season) VALUES("

#for i in range(len(weekStarts)):
for i in range(25,27):
    connection = sqlite3.connect('player_batting.db')
    cursor     = connection.cursor()

    weekStart = weekStarts[i]
    weekEnd   = weekEnds  [i]

    sbQuery = sbQueryPartOne + weekStart + sbQueryPartTwo + weekEnd + "\""

    cursor.execute(sbQuery)

    rows = cursor.fetchall()

    weeklyPlayerToSbs = {}

    for row in rows:
        curId          = row[0]
        curStolenBases = int(row[1])

        if curId in weeklyPlayerToSbs:
            prevStolenBases = weeklyPlayerToSbs[curId]
            weeklyPlayerToSbs[curId] = prevStolenBases + curStolenBases
        else:
            weeklyPlayerToSbs[curId] = curStolenBases

    cursor    .close()
    connection.close()

    weeklySbList = []

    for key in weeklyPlayerToSbs:
        sbs = weeklyPlayerToSbs[key]
        weeklySbList.append(sbs)

    weeklySbList.sort()

    for key in weeklyPlayerToSbs:
        sbs = weeklyPlayerToSbs[key]

        percentile = get_percentile(weeklySbList, sbs)

        formattedAddGradeStatement = addGradeStatement + str(i+1) + "," + "\"" + weekStart + "\",\"" + weekEnd + "\",\"" + key + "\",\"STOLEN_BASE\"," + str(sbs) + "," + str(percentile) + ", " + "2025" + ")"

        connection = sqlite3.connect('player_batting.db')
        cursor     = connection.cursor()

        cursor.execute(formattedAddGradeStatement)
        connection.commit()
        
        cursor    .close()
        connection.close()        

In [13]:
sbQuery = "SELECT id,sum(b_sb) FROM batters WHERE gametype = \"regular\" AND date BETWEEN \"2025-03-18\" AND \""

addGradeStatement = "INSERT INTO aggregate_weekly_grades (week_number,week_start,week_end,player_id,stat,aggregate_num,percentile,season) VALUES("

for i in range(25,27):
    weeklyPlayerToSbs = {}
    
    connection = sqlite3.connect('player_batting.db')
    cursor     = connection.cursor()

    weekStart = weekStarts[i]
    weekEnd   = weekEnds  [i]

    playerSbQuery = sbQuery + weekEnd + "\" GROUP BY id"

    cursor.execute(playerSbQuery)

    rows = cursor.fetchall()

    for row in rows:
        curId          = row[0]
        curStolenBases = int(row[1])

        weeklyPlayerToSbs[curId] = curStolenBases

    cursor    .close()
    connection.close()

    weeklySbList = []

    for key in weeklyPlayerToSbs:
        sbs = weeklyPlayerToSbs[key]
        weeklySbList.append(sbs)

    weeklySbList.sort()

    for key in weeklyPlayerToSbs:
        sbs = weeklyPlayerToSbs[key]

        percentile = get_percentile(weeklySbList, sbs)

        formattedAddGradeStatement = addGradeStatement + str(i+1) + "," + "\"" + weekStart + "\",\"" + weekEnd + "\",\"" + key + "\",\"STOLEN_BASE\"," + str(sbs) + "," + str(percentile) + ", " + "2025" + ")"

        connection = sqlite3.connect('player_batting.db')
        cursor     = connection.cursor()

        cursor.execute(formattedAddGradeStatement)
        connection.commit()
        
        cursor    .close()
        connection.close()        

In [ ]:
print(len(weeklyPlayerToSbs))

In [14]:
obpQueryPartOne = "SELECT id,b_h,b_w,b_hbp,b_ab,b_sf FROM batters WHERE b_pa != \"0\" AND b_ab != \"0\" AND gametype = \"regular\" AND date BETWEEN \""
obpQueryPartTwo = "\" AND \""

addGradeStatement = "INSERT INTO weekly_grades (week_number,week_start,week_end,player_id,stat,num,percentile,season) VALUES("

#for i in range(len(weekStarts)):
for i in range(25,27):
    connection = sqlite3.connect('player_batting.db')
    cursor     = connection.cursor()

    weekStart = weekStarts[i]
    weekEnd   = weekEnds  [i]

    obpQuery = obpQueryPartOne + weekStart + obpQueryPartTwo + weekEnd + "\""

    cursor.execute(obpQuery)

    rows = cursor.fetchall()

    weeklyPlayerToHits         = {}
    weeklyPlayerToWalks        = {}
    weeklyPlayerToHitByPitches = {}
    weeklyPlayerToAtBats       = {}
    weeklyPlayerToSacFlies     = {}
    
    for row in rows:
        playerId          = row[0]
        hits              = float(row[1])
        walks             = float(row[2])
        hitByPitches      = float(row[3])
        atBats            = float(row[4])
        sacFlies          = float(row[5])

        if playerId in weeklyPlayerToHits:
            curHits = weeklyPlayerToHits[playerId]
            weeklyPlayerToHits[playerId] = curHits + hits
        else:
            weeklyPlayerToHits[playerId] = hits
    
        if playerId in weeklyPlayerToWalks:
            curWalks = weeklyPlayerToWalks[playerId]
            weeklyPlayerToWalks[playerId] = curWalks + walks
        else:
            weeklyPlayerToWalks[playerId] = walks
    
        if playerId in weeklyPlayerToHitByPitches:
            curHitByPitches = weeklyPlayerToHitByPitches[playerId]
            weeklyPlayerToHitByPitches[playerId] = curHitByPitches + hitByPitches
        else:
            weeklyPlayerToHitByPitches[playerId] = hitByPitches
    
        if playerId in weeklyPlayerToAtBats:
            curAtBats = weeklyPlayerToAtBats[playerId]
            weeklyPlayerToAtBats[playerId] = curAtBats + atBats
        else:
            weeklyPlayerToAtBats[playerId] = atBats
    
        if playerId in weeklyPlayerToSacFlies:
            curSacFlies = weeklyPlayerToSacFlies[playerId]
            weeklyPlayerToSacFlies[playerId] = curSacFlies + sacFlies
        else:
            weeklyPlayerToSacFlies[playerId] = sacFlies
            
    cursor    .close()
    connection.close()

    weeklyObpList     = []
    weeklyPlayerToObp = {}

    for key in weeklyPlayerToHits:
        hits         = weeklyPlayerToHits        [key]
        walks        = weeklyPlayerToWalks       [key]
        hitByPitches = weeklyPlayerToHitByPitches[key]
        atBats       = weeklyPlayerToAtBats      [key]
        sacFlies     = weeklyPlayerToSacFlies    [key]

        obp = (hits + walks + hitByPitches) / (atBats + walks + hitByPitches + sacFlies)
        obp = round(obp, 3)
        
        weeklyPlayerToObp[key] = obp
        
        weeklyObpList.append(obp)

    weeklyObpList.sort()

    for key in weeklyPlayerToObp:
        obp = weeklyPlayerToObp[key]

        percentile = get_percentile(weeklyObpList, obp)

        formattedAddGradeStatement = addGradeStatement + str(i+1) + "," + "\"" + weekStart + "\",\"" + weekEnd + "\",\"" + key + "\",\"ON_BASE_PERCENTAGE\"," + str(obp) + "," + str(percentile) + ", " + "2025" + ")"

        connection = sqlite3.connect('player_batting.db')
        cursor     = connection.cursor()

        cursor.execute(formattedAddGradeStatement)
        connection.commit()
        
        cursor    .close()
        connection.close()        

In [15]:
obpQuery = "SELECT id,sum(b_h),sum(b_w),sum(b_hbp),sum(b_ab),sum(b_sf),sum(b_sh),sum(b_pa) FROM batters WHERE b_pa != \"0\" AND gametype = \"regular\" AND date BETWEEN \"2025-03-18\" AND \""

addGradeStatement = "INSERT INTO aggregate_weekly_grades (week_number,week_start,week_end,player_id,stat,aggregate_num,percentile,season) VALUES("

for i in range(25,27):
    aggregateWeeklyPlayerToHits         = {}
    aggregateWeeklyPlayerToWalks        = {}
    aggregateWeeklyPlayerToHitByPitches = {}
    aggregateWeeklyPlayerToAtBats       = {}
    aggregateWeeklyPlayerToSacFlies     = {}

    connection = sqlite3.connect('player_batting.db')
    cursor     = connection.cursor()

    weekStart = weekStarts[i]
    weekEnd   = weekEnds  [i]

    playerObpQuery = obpQuery + weekEnd + "\" GROUP BY id"

    cursor.execute(playerObpQuery)

    rows = cursor.fetchall()
    
    for row in rows:
        playerId          = row[0]
        hits              = float(row[1])
        walks             = float(row[2])
        hitByPitches      = float(row[3])
        atBats            = float(row[4])
        sacFlies          = float(row[5])
        sacHits           = int  (row[6])
        plateAppearences  = int  (row[7])

        if sacHits == plateAppearences:
            continue

        aggregateWeeklyPlayerToHits        [playerId] = hits
        aggregateWeeklyPlayerToWalks       [playerId] = walks
        aggregateWeeklyPlayerToHitByPitches[playerId] = hitByPitches
        aggregateWeeklyPlayerToAtBats      [playerId] = atBats
        aggregateWeeklyPlayerToSacFlies    [playerId] = sacFlies
            
    cursor    .close()
    connection.close()

    weeklyObpList     = []
    weeklyPlayerToObp = {}

    for key in aggregateWeeklyPlayerToHits:
        hits         = aggregateWeeklyPlayerToHits        [key]
        walks        = aggregateWeeklyPlayerToWalks       [key]
        hitByPitches = aggregateWeeklyPlayerToHitByPitches[key]
        atBats       = aggregateWeeklyPlayerToAtBats      [key]
        sacFlies     = aggregateWeeklyPlayerToSacFlies    [key]

        obp = (hits + walks + hitByPitches) / (atBats + walks + hitByPitches + sacFlies)
        obp = round(obp, 3)
        
        weeklyPlayerToObp[key] = obp
        
        weeklyObpList.append(obp)

    weeklyObpList.sort()

    for key in weeklyPlayerToObp:
        obp = weeklyPlayerToObp[key]

        percentile = get_percentile(weeklyObpList, obp)

        formattedAddGradeStatement = addGradeStatement + str(i+1) + "," + "\"" + weekStart + "\",\"" + weekEnd + "\",\"" + key + "\",\"ON_BASE_PERCENTAGE\"," + str(obp) + "," + str(percentile) + ", " + "2025" + ")"

        connection = sqlite3.connect('player_batting.db')
        cursor     = connection.cursor()

        cursor.execute(formattedAddGradeStatement)
        connection.commit()
        
        cursor    .close()
        connection.close()        

In [50]:
obpQuery = "SELECT id,team,sum(b_h),sum(b_w),sum(b_hbp),sum(b_ab),sum(b_sf),sum(b_sh),sum(b_pa) FROM batters WHERE b_pa != \"0\" AND gametype = \"regular\" AND date BETWEEN \"2025-03-18\" AND \""

updateGradeStatementPartOne   = "UPDATE aggregate_weekly_grades SET qualifying_percentile="
updateGradeStatementPartTwo   = " WHERE player_id=\""
updateGradeStatementPartThree = "\" AND stat=\"ON_BASE_PERCENTAGE\" and season=2025 and week_number="

for i in range(len(weekStarts)):
    aggregateWeeklyPlayerToHits             = {}
    aggregateWeeklyPlayerToWalks            = {}
    aggregateWeeklyPlayerToHitByPitches     = {}
    aggregateWeeklyPlayerToAtBats           = {}
    aggregateWeeklyPlayerToSacFlies         = {}
    aggregateWeeklyPlayerToPlateAppearences = {}
    aggregateWeeklyPlayerToTeam             = {}

    connection = sqlite3.connect('player_batting.db')
    cursor     = connection.cursor()

    weekStart = weekStarts[i]
    weekEnd   = weekEnds  [i]

    playerObpQuery = obpQuery + weekEnd + "\" GROUP BY id"

    cursor.execute(playerObpQuery)

    rows = cursor.fetchall()
    
    for row in rows:
        playerId          = row[0]
        team              = row[1]
        hits              = float(row[2])
        walks             = float(row[3])
        hitByPitches      = float(row[4])
        atBats            = float(row[5])
        sacFlies          = float(row[6])
        sacHits           = int  (row[7])
        plateAppearences  = int  (row[8])

        if sacHits == plateAppearences:
            continue

        aggregateWeeklyPlayerToHits            [playerId] = hits
        aggregateWeeklyPlayerToWalks           [playerId] = walks
        aggregateWeeklyPlayerToHitByPitches    [playerId] = hitByPitches
        aggregateWeeklyPlayerToAtBats          [playerId] = atBats
        aggregateWeeklyPlayerToSacFlies        [playerId] = sacFlies
        aggregateWeeklyPlayerToPlateAppearences[playerId] = plateAppearences
        aggregateWeeklyPlayerToTeam            [playerId] = team
                
    cursor    .close()
    connection.close()

    weeklyQualifiedObpList     = []
    weeklyQualifiedPlayerToObp = {}

    for key in aggregateWeeklyPlayerToHits:
        hits         = aggregateWeeklyPlayerToHits        [key]
        walks        = aggregateWeeklyPlayerToWalks       [key]
        hitByPitches = aggregateWeeklyPlayerToHitByPitches[key]
        atBats       = aggregateWeeklyPlayerToAtBats      [key]
        sacFlies     = aggregateWeeklyPlayerToSacFlies    [key]

        obp = (hits + walks + hitByPitches) / (atBats + walks + hitByPitches + sacFlies)
        obp = round(obp, 3)

        plateAppearences = aggregateWeeklyPlayerToPlateAppearences[key]
        playerTeam       = aggregateWeeklyPlayerToTeam            [key]

        teamGames = 0
        
        if playerTeam in teamToGamesPlayed[i]:
            teamGames = teamToGamesPlayed[i][playerTeam]

        paPerGame = 0.0
        
        if teamGames != 0:
            paPerGame = float(plateAppearences) / float(teamGames)

        if paPerGame >= 3.10:
            weeklyQualifiedPlayerToObp[key] = obp
        
            weeklyQualifiedObpList.append(obp)

    weeklyQualifiedObpList.sort()

    for key in weeklyQualifiedPlayerToObp:
        obp = weeklyQualifiedPlayerToObp[key]

        percentile = get_percentile(weeklyQualifiedObpList, obp)

        formattedUpdateGradeStatement = updateGradeStatementPartOne + str(percentile) + updateGradeStatementPartTwo + key + updateGradeStatementPartThree + str(i+1)

        connection = sqlite3.connect('player_batting.db')
        cursor     = connection.cursor()

        cursor.execute(formattedUpdateGradeStatement)
        connection.commit()
        
        cursor    .close()
        connection.close()        

In [16]:
rbiQueryPartOne = "SELECT id,b_rbi FROM batters WHERE b_pa != \"0\" AND gametype = \"regular\" AND date BETWEEN \""
rbiQueryPartTwo = "\" AND \""

addGradeStatement = "INSERT INTO weekly_grades (week_number,week_start,week_end,player_id,stat,num,percentile,season) VALUES("

#for i in range(len(weekStarts)):
for i in range(25,27):
    connection = sqlite3.connect('player_batting.db')
    cursor     = connection.cursor()

    weekStart = weekStarts[i]
    weekEnd   = weekEnds  [i]

    rbiQuery = rbiQueryPartOne + weekStart + rbiQueryPartTwo + weekEnd + "\""

    cursor.execute(rbiQuery)

    rows = cursor.fetchall()

    weeklyPlayerToRbis = {}
    
    for row in rows:
        playerId  = row[0]
        rbis      = int(row[1])

        if playerId in weeklyPlayerToRbis:
            curRbis = weeklyPlayerToRbis[playerId]
            weeklyPlayerToRbis[playerId] = curRbis + rbis
        else:
            weeklyPlayerToRbis[playerId] = rbis

    cursor    .close()
    connection.close()

    weeklyRbiList = []

    for key in weeklyPlayerToRbis:
        rbis = weeklyPlayerToRbis[key]
        weeklyRbiList.append(rbis)

    weeklyRbiList.sort()

    for key in weeklyPlayerToRbis:
        rbis = weeklyPlayerToRbis[key]

        percentile = get_percentile(weeklyRbiList, rbis)

        #(week_number,week_start,week_end,player_id,stat,num,grade)
        
        formattedAddGradeStatement = addGradeStatement + str(i+1) + "," + "\"" + weekStart + "\",\"" + weekEnd + "\",\"" + key + "\",\"RBI\"," + str(rbis) + "," + str(percentile) + ", " + "2025" + ")"

        connection = sqlite3.connect('player_batting.db')
        cursor     = connection.cursor()

        cursor.execute(formattedAddGradeStatement)
        connection.commit()
        
        cursor    .close()
        connection.close()        

In [17]:
rbiQuery = "SELECT id,sum(b_rbi) FROM batters WHERE b_pa != \"0\" AND gametype = \"regular\" AND date BETWEEN \"2025-03-18\" AND \""

addGradeStatement = "INSERT INTO aggregate_weekly_grades (week_number,week_start,week_end,player_id,stat,aggregate_num,percentile,season) VALUES("

for i in range(25,27):
    aggregateWeeklyPlayerToRbis = {}

    connection = sqlite3.connect('player_batting.db')
    cursor     = connection.cursor()

    weekStart = weekStarts[i]
    weekEnd   = weekEnds  [i]

    playerRbiQuery = rbiQuery + weekEnd + "\" GROUP BY id"

    cursor.execute(playerRbiQuery)

    rows = cursor.fetchall()
    
    for row in rows:
        playerId  = row[0]
        rbis      = int(row[1])

        aggregateWeeklyPlayerToRbis[playerId] = rbis

    cursor    .close()
    connection.close()

    weeklyRbiList = []

    for key in aggregateWeeklyPlayerToRbis:
        rbis = aggregateWeeklyPlayerToRbis[key]
        weeklyRbiList.append(rbis)

    weeklyRbiList.sort()

    for key in aggregateWeeklyPlayerToRbis:
        rbis = aggregateWeeklyPlayerToRbis[key]

        percentile = get_percentile(weeklyRbiList, rbis)
        
        formattedAddGradeStatement = addGradeStatement + str(i+1) + "," + "\"" + weekStart + "\",\"" + weekEnd + "\",\"" + key + "\",\"RBI\"," + str(rbis) + "," + str(percentile) + ", " + "2025" + ")"

        connection = sqlite3.connect('player_batting.db')
        cursor     = connection.cursor()

        cursor.execute(formattedAddGradeStatement)
        connection.commit()
        
        cursor    .close()
        connection.close()        

In [18]:
runQueryPartOne = "SELECT id,b_r FROM batters WHERE (b_pa != \"0\" OR b_r != \"0\") AND gametype = \"regular\" AND date BETWEEN \""
runQueryPartTwo = "\" AND \""

addGradeStatement = "INSERT INTO weekly_grades (week_number,week_start,week_end,player_id,stat,num,percentile,season) VALUES("

#for i in range(len(weekStarts)):
for i in range(25,27):
    connection = sqlite3.connect('player_batting.db')
    cursor     = connection.cursor()

    weekStart = weekStarts[i]
    weekEnd   = weekEnds  [i]

    runQuery = runQueryPartOne + weekStart + runQueryPartTwo + weekEnd + "\""

    cursor.execute(runQuery)

    rows = cursor.fetchall()

    weeklyPlayerToRuns = {}
    
    for row in rows:
        playerId  = row[0]
        runs      = int(row[1])

        if playerId in weeklyPlayerToRuns:
            curRuns = weeklyPlayerToRuns[playerId]
            weeklyPlayerToRuns[playerId] = curRuns + runs
        else:
            weeklyPlayerToRuns[playerId] = runs

    cursor    .close()
    connection.close()

    weeklyRunList = []

    for key in weeklyPlayerToRuns:
        runs = weeklyPlayerToRuns[key]
        weeklyRunList.append(runs)

    weeklyRunList.sort()

    for key in weeklyPlayerToRuns:
        runs = weeklyPlayerToRuns[key]

        percentile = get_percentile(weeklyRunList, runs)

        formattedAddGradeStatement = addGradeStatement + str(i+1) + "," + "\"" + weekStart + "\",\"" + weekEnd + "\",\"" + key + "\",\"RUN\"," + str(runs) + "," + str(percentile) + ", " + "2025" + ")"

        connection = sqlite3.connect('player_batting.db')
        cursor     = connection.cursor()

        cursor.execute(formattedAddGradeStatement)
        connection.commit()
        
        cursor    .close()
        connection.close()        

In [19]:
#CANNOT USE B_PA != 0 SINCE PLAYERS CAN SCORE AS PINCH RUNNERS

runsQuery = "SELECT id,sum(b_r) FROM batters WHERE (b_pa != \"0\" OR b_r != \"0\") AND gametype = \"regular\" AND date BETWEEN \"2025-03-18\" AND \""

addGradeStatement = "INSERT INTO aggregate_weekly_grades (week_number,week_start,week_end,player_id,stat,aggregate_num,percentile,season) VALUES("

for i in range(25,27):
    aggregateWeeklyPlayerToRuns = {}
    
    connection = sqlite3.connect('player_batting.db')
    cursor     = connection.cursor()

    weekStart = weekStarts[i]
    weekEnd   = weekEnds  [i]

    playerRunsQuery = runsQuery + weekEnd + "\" GROUP BY id"

    cursor.execute(playerRunsQuery)

    rows = cursor.fetchall()
    
    for row in rows:
        playerId  = row[0]
        runs      = int(row[1])

        aggregateWeeklyPlayerToRuns[playerId] = runs

    cursor    .close()
    connection.close()

    weeklyRunsList = []

    for key in aggregateWeeklyPlayerToRuns:
        runs = aggregateWeeklyPlayerToRuns[key]
        weeklyRunsList.append(runs)

    weeklyRunsList.sort()

    for key in aggregateWeeklyPlayerToRuns:
        runs = aggregateWeeklyPlayerToRuns[key]

        percentile = get_percentile(weeklyRunsList, runs)
        
        formattedAddGradeStatement = addGradeStatement + str(i+1) + "," + "\"" + weekStart + "\",\"" + weekEnd + "\",\"" + key + "\",\"RUN\"," + str(runs) + "," + str(percentile) + ", " + "2025" + ")"

        connection = sqlite3.connect('player_batting.db')
        cursor     = connection.cursor()

        cursor.execute(formattedAddGradeStatement)
        connection.commit()
        
        cursor    .close()
        connection.close()        

In [20]:
weekToOverallGrades      = {}
weekToOverallPercentile  = {}

runStat              = "RUN"
hrStat               = "HOME_RUN"
onBasePercentageStat = "ON_BASE_PERCENTAGE"
stolenBaseStat       = "STOLEN_BASE"
rbiStat              = "RBI"

partialAggregateStatQuery = "SELECT stat,player_id,aggregate_num FROM aggregate_weekly_grades WHERE week_number="
addGradeStatement         = "INSERT INTO aggregate_weekly_grades (week_number,week_start,week_end,player_id,stat,aggregate_num,percentile,season) VALUES("

playerIds = set()

for i in range(26,28):
    weekToOverallGrades    [i] = {}
    weekToOverallPercentile[i] = {}
    aggregateStatQuery = partialAggregateStatQuery + str(i)

    connection = sqlite3.connect('player_batting.db')
    cursor     = connection.cursor()

    cursor.execute(aggregateStatQuery)

    rows = cursor.fetchall()

    maxRuns = 0.0
    minRuns = 10000.0

    maxHrs = 0.0
    minHrs = 10000.0

    maxObp = 0.0
    minObp = 10000.0

    maxStolenBases = 0.0
    minStolenBases = 10000.0

    maxRbis = 0.0
    minRbis = 10000.0

    weekToOverallRuns        = {}
    weekToOverallRbis        = {}
    weekToOverallHomeRuns    = {}
    weekToOverallObp         = {}
    weekToOverallStolenBases = {}
    
    for row in rows:
        stat     = row[0]
        playerId = row[1]
        num      = float(row[2])

        playerIds.add(playerId)
        
        if stat == runStat:
            weekToOverallRuns[playerId] = num
            maxRuns = max(num, maxRuns)
            minRuns = min(num, minRuns)
        elif stat == hrStat:
            weekToOverallHomeRuns[playerId] = num
            maxHrs = max(num, maxHrs)
            minHrs = min(num, minHrs)
        elif stat == onBasePercentageStat:
            weekToOverallObp[playerId] = num
            maxObp = max(num, maxObp)
            minObp = min(num, minObp)
        elif stat == stolenBaseStat:
            weekToOverallStolenBases[playerId] = num
            maxStolenBases = max(num, maxStolenBases)
            minStolenBases = min(num, minStolenBases)
        elif stat == rbiStat:
            weekToOverallRbis[playerId] = num
            maxRbis = max(num, maxRbis)
            minRbis = min(num, minRbis)

    cursor    .close()
    connection.close()

    for playerId in playerIds:
        if playerId not in weekToOverallRuns:
            minRuns = 0.0
        if playerId not in weekToOverallHomeRuns:
            minHrs = 0.0
        if playerId not in weekToOverallObp:
            minObp = 0.0
        if playerId not in weekToOverallStolenBases:
            minStolenBases = 0.0
        if playerId not in weekToOverallRbis:
            minRbis = 0.0

    overallGrades = []

    print(f"{i}, {maxRuns}, {minRuns}, {maxHrs}, {minHrs}, {maxObp}, {minObp}, {maxStolenBases}, {minStolenBases}, {maxRbis}, {minRbis}")
    
    for playerId in playerIds:
        runs        = 0.0
        hrs         = 0.0
        obp         = 0.0
        stolenBases = 0.0
        rbis        = 0.0

        if playerId in weekToOverallRuns:
            runs = weekToOverallRuns[playerId]
        if playerId in weekToOverallHomeRuns:
            hrs = weekToOverallHomeRuns[playerId]
        if playerId in weekToOverallObp:
            obp = weekToOverallObp[playerId]
        if playerId in weekToOverallStolenBases:
            stolenBases = weekToOverallStolenBases[playerId]
        if playerId in weekToOverallRbis:
            rbis = weekToOverallRbis[playerId]

        normalizedRuns        = normalize(runs, maxRuns, minRuns)
        normalizedHrs         = normalize(hrs, maxHrs, minHrs)
        normalizedObp         = normalize(obp, maxObp, minObp)
        normalizedStolenBases = normalize(stolenBases, maxStolenBases, minStolenBases)
        normalizedRbis        = normalize(rbis, maxRbis, minRbis)

        overallGrade = normalizedRuns + normalizedHrs + normalizedObp + normalizedStolenBases + normalizedRbis

        weekToOverallGrades[i][playerId] = overallGrade

        overallGrades.append(overallGrade)
    
    overallGrades.sort()
    
    for playerId in playerIds:
        grade = weekToOverallGrades[i][playerId]

        percentile = get_percentile(overallGrades, grade)

        weekToOverallPercentile[i][playerId] = percentile

        formattedAddGradeStatement = addGradeStatement + str(i) + "," + "\"" + weekStart + "\",\"" + weekEnd + "\",\"" + playerId + "\",\"TOTAL\"," + str(grade) + "," + str(percentile) + ", " + "2025" + ")"

        connection = sqlite3.connect('player_batting.db')
        cursor     = connection.cursor()

        cursor.execute(formattedAddGradeStatement)
        connection.commit()
        
        cursor    .close()
        connection.close() 

26, 135.0, 0.0, 54.0, 0.0, 1.0, 0.0, 64.0, 0.0, 132.0, 0.0
27, 141.0, 0.0, 58.0, 0.0, 1.0, 0.0, 65.0, 0.0, 138.0, 0.0


In [21]:
runStat              = "RUN"
hrStat               = "HOME_RUN"
onBasePercentageStat = "ON_BASE_PERCENTAGE"
stolenBaseStat       = "STOLEN_BASE"
rbiStat              = "RBI"

partialAggregateStatQuery = "SELECT stat,player_id,num FROM weekly_grades WHERE season=2025 AND week_number="
addGradeStatement         = "INSERT INTO weekly_grades (week_number,week_start,week_end,player_id,stat,num,percentile,season) VALUES("

weeklyPlayerIds = set()

weeklyGrades     = {}
weeklyPercentile = {}

#for i in range(1,18):
for i in range(26,28):
    weeklyGrades    [i] = {}
    weeklyPercentile[i] = {}
    
    aggregateStatQuery = partialAggregateStatQuery + str(i)

    connection = sqlite3.connect('player_batting.db')
    cursor     = connection.cursor()

    cursor.execute(aggregateStatQuery)

    rows = cursor.fetchall()

    maxRuns = 0.0
    minRuns = 10000.0

    maxHrs = 0.0
    minHrs = 10000.0

    maxObp = 0.0
    minObp = 10000.0

    maxStolenBases = 0.0
    minStolenBases = 10000.0

    maxRbis = 0.0
    minRbis = 10000.0

    weekToOverallRuns        = {}
    weekToOverallRbis        = {}
    weekToOverallHomeRuns    = {}
    weekToOverallObp         = {}
    weekToOverallStolenBases = {}
    
    for row in rows:
        stat     = row[0]
        playerId = row[1]
        num      = float(row[2])

        weeklyPlayerIds.add(playerId)
        
        if stat == runStat:
            weekToOverallRuns[playerId] = num
            maxRuns = max(num, maxRuns)
            minRuns = min(num, minRuns)
        elif stat == hrStat:
            weekToOverallHomeRuns[playerId] = num
            maxHrs = max(num, maxHrs)
            minHrs = min(num, minHrs)
        elif stat == onBasePercentageStat:
            weekToOverallObp[playerId] = num
            maxObp = max(num, maxObp)
            minObp = min(num, minObp)
        elif stat == stolenBaseStat:
            weekToOverallStolenBases[playerId] = num
            maxStolenBases = max(num, maxStolenBases)
            minStolenBases = min(num, minStolenBases)
        elif stat == rbiStat:
            weekToOverallRbis[playerId] = num
            maxRbis = max(num, maxRbis)
            minRbis = min(num, minRbis)

    cursor    .close()
    connection.close()

    for playerId in weeklyPlayerIds:
        if playerId not in weekToOverallRuns:
            minRuns = 0.0
        if playerId not in weekToOverallHomeRuns:
            minHrs = 0.0
        if playerId not in weekToOverallObp:
            minObp = 0.0
        if playerId not in weekToOverallStolenBases:
            minStolenBases = 0.0
        if playerId not in weekToOverallRbis:
            minRbis = 0.0

    overallGrades = []

    print(f"{i}, {maxRuns}, {minRuns}, {maxHrs}, {minHrs}, {maxObp}, {minObp}, {maxStolenBases}, {minStolenBases}, {maxRbis}, {minRbis}")
    
    for playerId in weeklyPlayerIds:
        runs        = 0.0
        hrs         = 0.0
        obp         = 0.0
        stolenBases = 0.0
        rbis        = 0.0

        if playerId in weekToOverallRuns:
            runs = weekToOverallRuns[playerId]
        if playerId in weekToOverallHomeRuns:
            hrs = weekToOverallHomeRuns[playerId]
        if playerId in weekToOverallObp:
            obp = weekToOverallObp[playerId]
        if playerId in weekToOverallStolenBases:
            stolenBases = weekToOverallStolenBases[playerId]
        if playerId in weekToOverallRbis:
            rbis = weekToOverallRbis[playerId]

        normalizedRuns        = normalize(runs, maxRuns, minRuns)
        normalizedHrs         = normalize(hrs, maxHrs, minHrs)
        normalizedObp         = normalize(obp, maxObp, minObp)
        normalizedStolenBases = normalize(stolenBases, maxStolenBases, minStolenBases)
        normalizedRbis        = normalize(rbis, maxRbis, minRbis)

        overallGrade = normalizedRuns + normalizedHrs + normalizedObp + normalizedStolenBases + normalizedRbis

        weeklyGrades[i][playerId] = overallGrade

        overallGrades.append(overallGrade)
    
    overallGrades.sort()

    overallWeekStart = weekStarts[i-1]
    overallWeekEnd   = weekEnds  [i-1]
    
    for playerId in weeklyPlayerIds:
        grade = weeklyGrades[i][playerId]

        percentile = get_percentile(overallGrades, grade)

        weeklyPercentile[i][playerId] = percentile

        formattedAddGradeStatement = addGradeStatement + str(i) + "," + "\"" + overallWeekStart + "\",\"" + overallWeekEnd + "\",\"" + playerId + "\",\"TOTAL\"," + str(grade) + "," + str(percentile) + ", " + "2025" + ")"

        connection = sqlite3.connect('player_batting.db')
        cursor     = connection.cursor()

        cursor.execute(formattedAddGradeStatement)
        connection.commit()
        
        cursor    .close()
        connection.close() 

26, 9.0, 0.0, 5.0, 0.0, 0.75, 0.0, 3.0, 0.0, 12.0, 0.0
27, 9.0, 0.0, 4.0, 0.0, 1.0, 0.0, 4.0, 0.0, 12.0, 0.0


In [ ]:
fantasyRosterQuery = "SELECT catcher,first_base,second_base,third_base,short_stop,second_short_stop,first_third_base,outfield_one,outfield_two,outfield_three,outfield_four,outfield_five,util_one,util_two,bench_one,bench_two,bench_three,bench_four, bench_five,il_one,il_two,il_three FROM fantasy_rosters"

rosteredPlayers = set()

connection = sqlite3.connect('player_batting.db')
cursor     = connection.cursor()

cursor.execute(fantasyRosterQuery)

rows = cursor.fetchall()

for row in rows:
    for player in row:
        if player != "":
            rosteredPlayers.add(player)

cursor    .close()
connection.close()

topPlayersQuery = "select player_id,bios.FIRST,bios.LAST,num,percentile from weekly_grades inner join bios on weekly_grades.player_id = bios.PLAYERID where percentile > 90.0 and stat=\"TOTAL\" and season=2025 and week_number=17 "

for curRosteredPlayer in rosteredPlayers:
    topPlayersQuery += f"AND player_id != \"{curRosteredPlayer}\" "

topPlayersQuery += "ORDER BY percentile DESC"

connection = sqlite3.connect('player_batting.db')
cursor     = connection.cursor()

cursor.execute(topPlayersQuery)

rows = cursor.fetchall()

for row in rows:
    playerId = row[0]
    firstName = row[1]
    lastName = row[2]
    num = row[3]
    percentile = row[4]

    print(f"{playerId}, {firstName}, {lastName}, {num}, {percentile}")

cursor    .close()
connection.close()
    